<a href="https://colab.research.google.com/github/hasan-mahmud-sazid/Fintech-Propensity-Modelling-for-Enhanced-Campaign-Optimization/blob/main/Fintech_Propensity_Modelling_for_Enhanced_Campaign_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install pandas numpy scikit-learn smogn

In [3]:
import pandas as pd
import numpy as np
import smogn
from sklearn.model_selection import train_test_split

In [4]:
np.random.seed(42)
num_users = 10000

user_ids = [f"User_{i:05d}" for i in range (1,num_users+1)]

impressions = np.random.randint(100, 5001, size = num_users)

click_through_rate = np.random.uniform(0.01, 0.10, size = num_users)
click = (impressions *click_through_rate).astype(int)

session_time_avg = np.random.uniform(2.5, 45.0, size =num_users)

app_lunch_freq = np.random.randint(1,61, size = num_users)

In [5]:
base_conversion = (click*0.15)+(session_time_avg*0.05)+(app_lunch_freq*0.08)
noise = np.random.normal(0,1, size=num_users)
conversion_rate = base_conversion+noise

conversion_rate = np.clip(conversion_rate, 0, None)

In [6]:
conversion_rate = np.clip(conversion_rate, 0, None)
conversion_rate = (conversion_rate/conversion_rate.max())*100

In [7]:
df_simulated = pd.DataFrame({
    'user_id': user_ids,
    'impressions': impressions,
    'click': click,
    'session_time_avg': session_time_avg,
    'app_lunch_freq': app_lunch_freq,
    'conversion_rate': conversion_rate
})

df_simulated.head()

,user_id,impressions,click,session_time_avg,app_lunch_freq,conversion_rate
0,User_00001,960,11,33.077768,41,8.881055
1,User_00002,3872,155,42.984904,13,32.933342
2,User_00003,3192,312,13.724692,50,64.457531
3,User_00004,566,50,6.227110,25,13.800004
4,User_00005,4526,179,22.711728,7,36.254726


In [8]:
df_simulated.to_csv('fintec_simulated_data.csv', index = False)

In [9]:
df = pd.read_csv('fintec_simulated_data.csv')

df_ml = df.drop(columns=['user_id'])
df_ml.head()

,impressions,click,session_time_avg,app_lunch_freq,conversion_rate
0,960,11,33.077768,41,8.881055
1,3872,155,42.984904,13,32.933342
2,3192,312,13.724692,50,64.457531
3,566,50,6.227110,25,13.800004
4,4526,179,22.711728,7,36.254726


In [11]:
print("Running SMOGN to balance the dataset.....")

df_smogn = smogn.smoter(
    data= df_ml, # Changed from df_ml.sample(n=2000, random_state=42).reset_index(drop=True)
    y = 'conversion_rate',
    k = 5,
    samp_method = 'extreme',
    rel_thres=0.80
)

Running SMOGN to balance the dataset.....


r_index: 100%|##########| 449/449 [00:00<00:00, 1335.01it/s]


In [12]:
print(f"Original Sample Size: 2000 | SMOGN Resampled Size: {len(df_smogn)}")

Original Sample Size: 2000 | SMOGN Resampled Size: 17936


In [13]:
df_smogn.head()

,impressions,click,session_time_avg,app_lunch_freq,conversion_rate
0,4571.0,369.0,16.501646,11.0,72.908371
1,4581.0,359.0,23.803165,19.0,72.519005
2,4579.0,350.0,27.611844,14.0,72.763619
3,4584.0,367.0,35.054137,31.0,73.299942
4,4572.0,366.0,17.831851,11.0,72.908371


In [14]:
X= df_ml.drop(columns= ['conversion_rate'])
y=df_ml['conversion_rate']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [19]:
X_train.head()

,impressions,click,session_time_avg,app_lunch_freq
9254,3951,361,17.791116,60
1561,986,86,44.605766,2
1670,113,5,22.198817,26
6087,1645,162,5.429976,45
6669,2697,136,10.198986,17


In [20]:
X_train_smogn = df_smogn.drop(columns=['conversion_rate'])
y_train_smogn = df_smogn['conversion_rate']

print("Data preprocessing and SMOGN split succesfully completed!")

Data preprocessing and SMOGN split succesfully completed!


In [21]:
X_train_smogn

,impressions,click,session_time_avg,app_lunch_freq
0,4571.0,369.0,16.501646,11.0
1,4581.0,359.0,23.803165,19.0
2,4579.0,350.0,27.611844,14.0
3,4584.0,367.0,35.054137,31.0
4,4572.0,366.0,17.831851,11.0
...,...,...,...,...
9995,1307.0,93.0,29.693190,45.0
9996,2242.0,157.0,6.513193,23.0
9997,884.0,20.0,28.855829,46.0
9998,2796.0,115.0,24.319786,12.0


In [22]:
y_train_smogn

,conversion_rate
0,72.908371
1,72.519005
2,72.763619
3,73.299942
4,72.908371
...,...
9995,24.178620
9996,31.186854
9997,9.189501
9998,27.285347


In [24]:
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error

print("Training Baseline XGBoost Model ..")
xgb_baseline = xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
xgb_baseline.fit(X_train, y_train)

Training Baseline XGBoost Model ..


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [26]:
y_pred_baseline=xgb_baseline.predict(X_test)
r2_baseline= r2_score(y_test, y_pred_baseline)
rmse_baseline=np.sqrt(mean_squared_error(y_test, y_pred_baseline))
print(f"R2 Score: {r2_baseline}")
print(f"RMSE: {rmse_baseline}")

R2 Score: 0.9960733234205765
RMSE: 1.297130788939433


In [27]:
print("Training SMOGN Optimized XGBoost Model..")
xgb_smogn = xgb.XGBRegressor(n_estimators=150, max_depth=6, learning_rate = 0.05, random_state=42)
xgb_smogn.fit(X_train_smogn, y_train_smogn)


y_pred_smogn = xgb_smogn.predict(X_test)
r2_smogn = r2_score(y_test, y_pred_smogn)
rmse_smogn = np.sqrt(mean_squared_error(y_test, y_pred_smogn))

print("\n"+"="*30)
print("Model Performance Comparison")
print("="*30)
print(f"Baseline XGBoost  -> R² Score: {r2_baseline:.4f} | RMSE: {rmse_baseline:.4f}")
print(f"SMOGN AI XGBoost -> R² Score: {r2_smogn:.4f} | RMSE: {rmse_smogn:.4f}")
print("="*30)

Training SMOGN Optimized XGBoost Model..

Model Performance Comparison
Baseline XGBoost  -> R² Score: 0.9961 | RMSE: 1.2971
SMOGN AI XGBoost -> R² Score: 0.9970 | RMSE: 1.1378
